# Lecture 13 — ML Interatomic Potentials & Foundation-Model Fine-Tuning

**PHYG004 / PHY5006, 2026 Spring · Sogang University**  
Prof. Young Woo Choi

---

## Learning goals
1. Understand the formalism of **NequIP**, **MACE**, and **MatterSim**.
2. Load and run three production-grade pretrained models — **MACE-MP-0**, **SevenNet-0**, **MatterSim-1M** — through a unified ASE `Calculator` interface.
3. **Benchmark against ground truth**: pull real relaxed Si structures from the **Materials Project** and compare predicted energies, forces, and bulk modulus to PBE-DFT reference values.
4. Run a short MD simulation with a foundation potential.
5. **Fine-tune** MACE-MP-0 on a small Si dataset and measure how its bulk modulus moves toward the DFT / experimental value.

## Prerequisites
- Lecture 11 — Equivariance & Symmetry in Neural Networks
- Lecture 12 — Equivariant Networks with `e3nn-jax`

> **Runtime tip:** in Colab, switch to **Runtime → Change runtime type → T4 GPU** before installing. Most cells run on CPU, but MD and MatterSim inference are noticeably faster on GPU.

<!-- lecture13-visual:start:title-pes -->
<div align="center">
  <img src="images/ai/01_pes_bridge.webp" width="780"/>
  <br><em>ML potentials bridge high-accuracy electronic-structure data and long-time atomistic simulation.</em>
</div>
<!-- lecture13-visual:end:title-pes -->



## 0. Setup

MACE currently requires `e3nn==0.4.4`, while recent MatterSim releases declare `e3nn>=0.5.0`. The install cell below uses `uv pip` and keeps the MACE-compatible `e3nn` while installing MatterSim without replacing it. Run the setup cell once in a fresh runtime.


In [ ]:
# ~3-5 min on a fresh Colab; longer if it pulls a CUDA-matched torch
import subprocess
import sys

try:
    import uv  # noqa: F401
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])

def uv_pip_install(*packages: str) -> None:
    subprocess.check_call([
        sys.executable, "-m", "uv", "pip", "install",
        "--python", sys.executable,
        *packages,
    ])

# MACE pins e3nn==0.4.4. Install this stack first so the compatible e3nn wins.
uv_pip_install(
    "mace-torch==0.3.16",
    "sevenn==0.10.4",
    "ase==3.28.0",
    "matplotlib==3.10.9",
)

# MatterSim runtime dependencies, excluding e3nn so MACE checkpoints still load.
uv_pip_install(
    "azure-identity", "azure-storage-blob", "deprecated",
    "atomate2", "emmet-core", "loguru", "mp-api",
    "pydantic>=2.9.2", "pymatgen", "seekpath", "phonopy", "phono3py",
    "torch-runstats", "torchaudio", "torchvision", "wandb",
)
uv_pip_install("--no-deps", "mattersim==1.2.3")

In [ ]:
import os
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")

import numpy as np
import torch
import matplotlib.pyplot as plt
import time

from ase import Atoms, units
from ase.build import bulk, molecule

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch:  {torch.__version__}")
print(f"device: {device}")
if device == "cuda":
    print(f"GPU:    {torch.cuda.get_device_name(0)}")

### Materials Project API key (for the real-data sections)

Sections 8b and 9b below query the **Materials Project** for real relaxed Si
structures and their PBE-DFT reference properties. This needs a free API key.

1. Sign in at <https://next-gen.materialsproject.org/> and copy your key from
   the dashboard (**API** tab).
2. In Colab, set it as an environment variable in the cell below (or use the
   Colab *Secrets* panel and read it with `userdata.get`).

If you skip this, the foundation-model comparison and MD cells still run; only
the `mp-api` ground-truth comparisons are skipped.


In [ ]:
# Materials Project API key — paste yours here (or use Colab Secrets).
# Leave empty to skip the mp-api ground-truth comparisons.
import os

MP_API_KEY = os.environ.get("MP_API_KEY", "")  # <-- set your key, e.g. "abcd...1234"

# Optional Colab Secrets path:
# from google.colab import userdata
# MP_API_KEY = userdata.get("MP_API_KEY")

os.environ["MP_API_KEY"] = MP_API_KEY
HAVE_MP_KEY = bool(MP_API_KEY)
print("Materials Project key set:", HAVE_MP_KEY)

## 1. Recap — what we want from a potential

For $N$ atoms with positions $\{\mathbf{R}_i\}$ and species $\{Z_i\}$, an interatomic potential is a scalar function
$$
E\!\left(\{\mathbf{R}_i, Z_i\}\right) \in \mathbb{R}, \qquad \mathbf{F}_i = -\nabla_{\mathbf{R}_i} E .
$$

**Required symmetries**
- Translation invariance: $E(\{\mathbf{R}_i + \mathbf{t}\}) = E(\{\mathbf{R}_i\})$.
- $O(3)$ invariance of $E$; forces transform **equivariantly** under rotations.
- Permutation invariance over atoms of the same species.

**Locality** — decompose the energy into per-atom contributions within a cutoff sphere of radius $r_c$:
$$
E = \sum_i E_i\!\left(\mathcal{N}(i)\right), \qquad \mathcal{N}(i) = \{ j : |\mathbf{R}_j - \mathbf{R}_i| < r_c \}.
$$

Modern MLPs differ mainly in **how the local environment is encoded** while preserving these symmetries by construction.

<!-- lecture13-visual:start:local-environment -->
<div align="center">
  <img src="images/diagrams/local_environment.png" width="820"/>
  <br><em>Locality converts a many-atom PES into a size-extensive sum of atomic energy contributions.</em>
</div>
<!-- lecture13-visual:end:local-environment -->



## 2. NequIP — E(3)-equivariant message passing

Batzner *et al.*, *Nat. Commun.* **13**, 2453 (2022).

**Features are direct sums of $SO(3)$ irreps**
$$
h_i \;=\; \bigoplus_{l} h_i^{(l)} \;\in\; \bigoplus_l \mathbb{R}^{2l+1}.
$$
Scalars live in $l{=}0$, vectors in $l{=}1$, …

**Equivariant message** from $j$ to $i$:
$$
m_{ij} \;=\; \sum_{l_f,\, l_s,\, l_o} R_{l_f l_s l_o}(r_{ij})\,\Big[\, Y^{(l_f)}(\hat{\mathbf{r}}_{ij}) \otimes_{l_o} h_j^{(l_s)} \,\Big]
$$
- $R_{l_f l_s l_o}$ : radial MLP on $r_{ij}$ (with smooth cutoff). **One independent radial MLP per allowed path** $(l_f, l_s, l_o)$ — i.e. per combination of the spherical-harmonic order $l_f$, the source-feature order $l_s$, and the output order $l_o$.
- $Y^{(l_f)}$ : real spherical harmonics — angular information.
- $h_j^{(l_s)}$ : the $l_s$ irrep block of the neighbor's feature; the source-feature order $l_s$ runs over all irreps carried by $h_j$.
- $\otimes_{l_o}$ : Clebsch–Gordan tensor product of $Y^{(l_f)}$ and $h_j^{(l_s)}$, projected onto output irrep $l_o$. A path is allowed only when $|l_f - l_s| \le l_o \le l_f + l_s$.

**Equivariant update**
$$
h_i \;\leftarrow\; \sigma\!\Big(W\,h_i + \sum_{j \in \mathcal{N}(i)} m_{ij}\Big),
$$
where $\sigma$ acts as a regular nonlinearity on the $l{=}0$ part and as a *gated* activation on $l>0$ parts so equivariance is preserved.

**Energy** comes from the invariant part:
$$
E_i \;=\; \mathrm{MLP}\!\left( h_i^{(0)} \right), \qquad E \;=\; \sum_i E_i.
$$

Because every operation is a CG tensor product of spherical harmonics with equivariant features, NequIP is **provably $E(3)$-equivariant by construction**. Empirically, this gives strong data efficiency — kcal/mol accuracy can be reached with $\sim 10^3$ DFT structures.

<!-- lecture13-visual:start:nequip -->
<table style="width:100%; border:0;">
<tr>
<td align="center" style="border:0; padding:8px;"><img src="images/ai/02_equivariant_message.webp" width="360"/><br><em>Visual intuition: geometric features rotate together with the atomic environment.</em></td>
<td align="center" style="border:0; padding:8px;"><img src="images/diagrams/nequip_message.png" width="360"/><br><em>Mechanism: radial filters, spherical harmonics, and tensor products form equivariant messages.</em></td>
</tr>
</table>
<!-- lecture13-visual:end:nequip -->



## 3. MACE — many-body equivariant messages

Batatia *et al.*, *NeurIPS* (2022); MACE-MP: Batatia *et al.*, arXiv:2401.00096 (2024).

**Step 1 — Atomic basis** (two-body, linear in the environment, just like NequIP)
$$
A_i^{(k,\,l)} \;=\; \sum_{j \in \mathcal{N}(i)} R_k(r_{ij})\, Y^{(l)}(\hat{\mathbf{r}}_{ij}) \otimes h_j .
$$

**Step 2 — Many-body features** via tensor products of $A$'s
$$
B_i^{(\nu)} \;=\; \sum_{k_1,\dots,k_\nu}
\Big( A_i^{(k_1)} \otimes A_i^{(k_2)} \otimes \cdots \otimes A_i^{(k_\nu)} \Big)_{\text{coupled to } l_o} .
$$
This is the equivariant generalisation of the **Atomic Cluster Expansion** (Drautz, 2019). With body order $\nu$ per layer and $L$ layers, the effective body order reaches $(\nu{+}1)^L$ — two layers with $\nu{=}3$ already capture up to 16-body interactions.

**Energy** is a sum of contributions from each layer’s invariant features:
$$
E \;=\; \sum_i \sum_{L'} W^{(L')} \cdot B_i^{(L',\, l=0)} .
$$

**Why MACE is fast.** High effective body order *per layer* means **fewer message-passing layers** are needed → less indirect communication → lower latency on large boxes. MACE-MP-0 is a 2-layer, $\nu{=}3$ model trained on the MPtrj subset of Materials Project (~$1.6 \times 10^6$ DFT structures, 89 elements).

<!-- lecture13-visual:start:mace -->
<table style="width:100%; border:0;">
<tr>
<td align="center" style="border:0; padding:8px;"><img src="images/ai/03_mace_many_body.webp" width="360"/><br><em>Visual intuition: one local environment contains multiple coupled many-body motifs.</em></td>
<td align="center" style="border:0; padding:8px;"><img src="images/diagrams/mace_body_order.png" width="360"/><br><em>MACE packs higher body order into fewer message-passing layers.</em></td>
</tr>
</table>
<!-- lecture13-visual:end:mace -->



## 4. SevenNet — distributed NequIP

Park *et al.* (Seoul National University). Code: [github.com/MDIL-SNU/SevenNet](https://github.com/MDIL-SNU/SevenNet).

- E(3)-equivariant message passing in the NequIP / Allegro family.
- **ZBL short-range repulsion** baked in, which makes MD stable under aggressive perturbations or high pressure.
- Distributed inference across multiple GPUs — practical for $> 10^5$-atom MD.
- **SevenNet-0** is the universal MPtrj-trained checkpoint — comparable scope to MACE-MP-0.
- Variants: **SevenNet-MF** (multi-fidelity training), **SevenNet-l3i5** (deeper), **SevenNet-D3** (with Grimme-D3 dispersion).

Think of SevenNet as a Korean, HPC-friendly cousin of NequIP.

<!-- lecture13-visual:start:sevennet -->
<div align="center">
  <img src="images/diagrams/sevennet_domain.png" width="760"/>
  <br><em>Distributed inference: domain decomposition plus boundary-neighbor communication.</em>
</div>
<!-- lecture13-visual:end:sevennet -->



## 5. MatterSim — broad-coverage foundation model

Yang *et al.*, *MatterSim: A Deep Learning Atomistic Model Across Elements, Temperatures and Pressures*, arXiv:2405.04967 (2024). Microsoft Research.

- **Architecture** — M3GNet-based (graph + bond-angle features) for the 1M model; EquiformerV2 variant for the 5M model.
- **Training set** — $\sim 1.7 \times 10^7$ structures generated by **active learning** across $(T, P)$ space:
  - Temperature: $0 - 5000$ K
  - Pressure: $0 - 1000$ GPa
  - Bulk, surfaces, liquids, amorphous, defects.
- **Checkpoints** — `MatterSim-v1.0.0-1M` (fast) and `MatterSim-v1.0.0-5M` (more accurate).
- Designed for **downstream property prediction** (phonons, free energies, EOS) with fine-tuning hooks.

Compared with MACE-MP / SevenNet, MatterSim trades some near-equilibrium accuracy for **dramatically broader thermodynamic coverage**.

<!-- lecture13-visual:start:mattersim -->
<table style="width:100%; border:0;">
<tr>
<td align="center" style="border:0; padding:8px;"><img src="images/ai/04_foundation_models.webp" width="360"/><br><em>Foundation potentials learn reusable atomistic representations across many materials classes.</em></td>
<td align="center" style="border:0; padding:8px;"><img src="images/diagrams/mattersim_tp.png" width="360"/><br><em>MatterSim emphasizes broad active-learning coverage over temperature and pressure.</em></td>
</tr>
</table>
<!-- lecture13-visual:end:mattersim -->



## 6. Foundation potential ecosystem (cheat sheet)

| Model | Architecture | Training set | Scope | License |
|-------|--------------|--------------|-------|---------|
| **CHGNet** | GNN + magnetic moments | MPtrj | inorganic | BSD-3 |
| **M3GNet** | GNN (bond + angle) | MPF.2021 | inorganic | BSD-3 |
| **MACE-MP-0** | Equivariant MPNN + ACE | MPtrj | inorganic, 89 elements | MIT |
| **MACE-OFF** | Equivariant MPNN + ACE | SPICE + extras | organic / drug-like | MIT |
| **SevenNet-0** | E(3) MPNN | MPtrj | inorganic | GPL-3 |
| **MatterSim-1M / 5M** | M3GNet / EqV2 | active-learned, 17M | broad $(T,P)$ | MIT |
| **Allegro** | Local equivariant NN | task-specific | trained per system | MIT |
| **ORB** | Graph transformer | MPtrj | inorganic | non-commercial |
| **EquiformerV2** | Transformer on irreps | OC20 / OC22 | catalysis | MIT |

Benchmarks: [Matbench Discovery](https://matbench-discovery.materialsproject.org/). Be careful — **leaderboard accuracy is not the same as usefulness for your science problem**.

<!-- lecture13-visual:start:ecosystem -->
<div align="center">
  <img src="images/diagrams/ecosystem_map.png" width="820"/>
  <br><em>Model selection is a scope-throughput-accuracy tradeoff, not a leaderboard-only decision.</em>
</div>
<!-- lecture13-visual:end:ecosystem -->



## 7. Hands-on A — load three calculators

All three models expose an ASE `Calculator`, so `atoms.get_potential_energy()` works uniformly once a calculator is attached.


In [ ]:
# --- MACE-MP-0 ---
from mace.calculators import mace_mp

mace_calc = mace_mp(
    model="medium",          # "small" / "medium" / "large"
    default_dtype="float32",
    device=device,
    dispersion=False,
)
print("MACE-MP-0 loaded.")

In [ ]:
# --- SevenNet-0 ---
from sevenn.calculator import SevenNetCalculator

sevennet_calc = SevenNetCalculator(model="7net-0", device=device)
print("SevenNet-0 loaded.")

In [ ]:
# --- MatterSim-1M ---
from mattersim.forcefield import MatterSimCalculator

mattersim_calc = MatterSimCalculator(
    load_path="MatterSim-v1.0.0-1M.pth",   # downloaded automatically on first call
    device=device,
)
print("MatterSim-1M loaded.")

## 8. Hands-on B — single-point on a perturbed Si supercell

We rattle a $2\times 2\times 2$ diamond-Si supercell (64 atoms) and compare **energies, forces, and timing** across the three models.

> Note: each model uses its own energy reference, so **absolute energies are not directly comparable**. Forces are.

<!-- lecture13-visual:start:single-point -->
<div align="center">
  <img src="images/diagrams/si_supercell.png" width="820"/>
  <br><em>The same perturbed Si supercell is passed through three ASE calculators.</em>
</div>
<!-- lecture13-visual:end:single-point -->



In [ ]:
rng = np.random.default_rng(42)

si = bulk("Si", "diamond", a=5.43, cubic=True).repeat((2, 2, 2))  # 64 atoms
si.rattle(stdev=0.05, seed=42)
print(f"System: {len(si)} Si atoms, cell lengths = {si.cell.lengths().round(2)} Å")

def evaluate(calc, name, atoms):
    a = atoms.copy()
    a.calc = calc
    t0 = time.perf_counter()
    E = a.get_potential_energy()
    F = a.get_forces()
    dt = time.perf_counter() - t0
    fmax = np.linalg.norm(F, axis=1).max()
    print(f"{name:>14s}:  E = {E:9.3f} eV   |F|max = {fmax:5.2f} eV/Å   ({dt*1000:6.1f} ms)")
    return E, F

E_mace,      F_mace      = evaluate(mace_calc,      "MACE-MP-0",    si)
E_sevenn,    F_sevenn    = evaluate(sevennet_calc,  "SevenNet-0",   si)
E_mattersim, F_mattersim = evaluate(mattersim_calc, "MatterSim-1M", si)

In [ ]:
def force_rmse(A, B):
    return float(np.sqrt(np.mean((A - B) ** 2)))

print(f"RMSE(MACE,     SevenNet)  = {force_rmse(F_mace,    F_sevenn):.4f} eV/Å")
print(f"RMSE(MACE,     MatterSim) = {force_rmse(F_mace,    F_mattersim):.4f} eV/Å")
print(f"RMSE(SevenNet, MatterSim) = {force_rmse(F_sevenn,  F_mattersim):.4f} eV/Å")

## 8b. Hands-on B+ — real ground truth from the Materials Project

The force RMSEs above only tell us how much the three models **disagree with each
other** — there is no ground truth. Let us fix that. We pull a *real* relaxed
Si structure (mp-149, diamond Si) from the **Materials Project**, evaluate the
three foundation models on it, and compare against the MP **PBE-DFT** reference.

> Physicist's framing: this is the same logic as validating a model Hamiltonian
> against a first-principles calculation before trusting it on new phase points.

**Helper:** `get_mp_structure` queries an `mp-id`, returns an ASE `Atoms` object
together with the MP DFT formation energy per atom and (if available) the DFT
bulk modulus.


In [ ]:
from typing import Optional, Tuple

def get_mp_structure(mp_id: str) -> Tuple[Optional["Atoms"], dict]:
    """Query Materials Project for a relaxed structure + DFT reference props.

    Returns (ase_atoms, info) where info has keys:
        formation_energy_per_atom (eV), energy_per_atom (eV),
        bulk_modulus_vrh (GPa, may be None), volume_per_atom (A^3),
        symbol, mp_id.
    Returns (None, {}) if no API key is configured.
    """
    if not HAVE_MP_KEY:
        print("No MP_API_KEY set — skipping mp-api query.")
        return None, {}

    from mp_api.client import MPRester
    from pymatgen.io.ase import AseAtomsAdaptor

    with MPRester(MP_API_KEY) as mpr:
        doc = mpr.materials.summary.get_data_by_id(
            mp_id,
            fields=[
                "structure", "formation_energy_per_atom",
                "energy_per_atom", "bulk_modulus", "volume", "nsites",
                "formula_pretty",
            ],
        )
    struct = doc.structure
    atoms = AseAtomsAdaptor.get_atoms(struct)
    print(f"{mp_id}: {doc.formula_pretty}, {len(atoms)} atoms, "
          f"cell lengths = {atoms.cell.lengths().round(3)} A")

    bulk_mod = None
    if getattr(doc, "bulk_modulus", None):
        # MP returns {"voigt":..,"reuss":..,"vrh":..} in GPa
        bulk_mod = doc.bulk_modulus.get("vrh")

    info = {
        "mp_id": mp_id,
        "symbol": doc.formula_pretty,
        "formation_energy_per_atom": doc.formation_energy_per_atom,
        "energy_per_atom": doc.energy_per_atom,
        "bulk_modulus_vrh": bulk_mod,
        "volume_per_atom": doc.volume / doc.nsites,
    }
    print(f"  MP DFT  E_form = {info['formation_energy_per_atom']:.4f} eV/atom, "
          f"V0 = {info['volume_per_atom']:.3f} A^3/atom, "
          f"B0(VRH) = {bulk_mod}")
    return atoms, info

In [ ]:
# --- Checkpoint A: model comparison against MP DFT ---------------------------
# TODO(student): change mp_id to another Si polymorph (e.g. "mp-149" diamond Si,
#                "mp-1014224" / "mp-92" beta-tin Si) and re-run.
si_mp, si_mp_info = get_mp_structure("mp-149")  # diamond Si primitive cell

if si_mp is not None:
    print("\nForeground-model single-point energies on the MP-relaxed cell:")
    print(f"{'Model':>14s}  {'E/atom (eV)':>12s}  {'|F|max (eV/A)':>13s}")
    print("-" * 44)
    model_results = {}
    for name, calc in [
        ("MACE-MP-0",    mace_calc),
        ("SevenNet-0",   sevennet_calc),
        ("MatterSim-1M", mattersim_calc),
    ]:
        a = si_mp.copy()
        a.calc = calc
        E = a.get_potential_energy() / len(a)
        F = a.get_forces()
        fmax = float(np.linalg.norm(F, axis=1).max())
        model_results[name] = {"E_per_atom": E, "fmax": fmax}
        print(f"{name:>14s}  {E:12.4f}  {fmax:13.4f}")

    # The MP cell is already relaxed at the PBE minimum, so |F| should be small.
    # Verification (Checkpoint A): on an in-distribution MPtrj structure, a model
    # that learned PBE forces should predict near-zero residual force.
    print("\nVerification — residual force on the DFT-relaxed cell "
          "(should be small if the model matches PBE):")
    PASS_TH = 0.15  # eV/A
    for name, r in model_results.items():
        ok = r["fmax"] < PASS_TH
        print(f"  {name:>14s}: |F|max = {r['fmax']:.4f} eV/A  "
              f"-> {'PASS' if ok else 'CHECK'} (threshold {PASS_TH} eV/A)")
    print(f"\nMP DFT formation energy = "
          f"{si_mp_info['formation_energy_per_atom']:.4f} eV/atom "
          f"(diamond Si is the reference phase, so ~0).")
else:
    print("Skipped Checkpoint A (no MP key).")

## 9. Hands-on C — equation of state of bulk Si

Sweep the cell volume and recompute energy at each scale. Subtract each model's own minimum so the three curves can be overlaid. A quadratic fit near the minimum gives the **bulk modulus** $B_0 = V \, d^2 E / dV^2$.

<!-- lecture13-visual:start:eos -->
<div align="center">
  <img src="images/diagrams/eos_concept.png" width="820"/>
  <br><em>The curvature of the energy-volume curve determines the bulk modulus.</em>
</div>
<!-- lecture13-visual:end:eos -->



In [ ]:
si0 = bulk("Si", "diamond", a=5.43, cubic=True)

scales = np.linspace(0.92, 1.10, 11)
volumes = []
energies = {"MACE-MP-0": [], "SevenNet-0": [], "MatterSim-1M": []}

for s in scales:
    a = si0.copy()
    a.set_cell(a.cell * s, scale_atoms=True)
    volumes.append(a.get_volume() / len(a))   # Å^3 / atom
    for name, calc in [
        ("MACE-MP-0",    mace_calc),
        ("SevenNet-0",   sevennet_calc),
        ("MatterSim-1M", mattersim_calc),
    ]:
        a.calc = calc
        energies[name].append(a.get_potential_energy() / len(a))   # eV / atom

volumes = np.array(volumes)
for name in energies:
    energies[name] = np.array(energies[name]) - np.min(energies[name])

In [ ]:
plt.figure(figsize=(6.5, 4.2))
for name, e in energies.items():
    plt.plot(volumes, e, "o-", label=name)
plt.xlabel("Volume per atom (Å³)")
plt.ylabel("E − E$_\\mathrm{min}$  (eV/atom)")
plt.title("Bulk Si — EOS from three pretrained MLPs")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()

In [ ]:
# Parabolic fit gives V0, E0, and B0 = V * d^2E/dV^2.
# 1 eV/Å^3 = 160.21766208 GPa.

def fit_eos(V, E):
    a, b, c = np.polyfit(V, E, 2)
    V0 = -b / (2 * a)
    E0 = a * V0 ** 2 + b * V0 + c
    B0 = 2 * a * V0 * 160.21766208
    return V0, E0, B0

print(f"{'Model':<15s} {'V0 (Å³/at)':>12s} {'B0 (GPa)':>10s}")
print("-" * 40)
for name, e in energies.items():
    V0, E0, B0 = fit_eos(volumes, e)
    print(f"{name:<15s} {V0:>12.3f} {B0:>10.1f}")
print("-" * 40)
print(f"{'Experiment':<15s} {20.0:>12.3f} {99.0:>10.1f}")

### A reusable EOS harness

So that we can re-use exactly the same equation-of-state measurement for the
**fine-tuned** model later, wrap the sweep + parabolic fit into two helpers.
This reproduces the numbers from Section 9 but as callable functions.


In [ ]:
def run_eos(calculators: dict, a0: float = 5.43,
            scales=np.linspace(0.92, 1.10, 11)) -> tuple:
    """Energy-volume sweep of diamond Si for a dict {name: ASE calculator}.

    Returns (volumes [A^3/atom], energies {name: eV/atom, min-subtracted}).
    """
    si0 = bulk("Si", "diamond", a=a0, cubic=True)
    vols, en = [], {name: [] for name in calculators}
    for s in scales:
        a = si0.copy()
        a.set_cell(a.cell * s, scale_atoms=True)
        vols.append(a.get_volume() / len(a))
        for name, calc in calculators.items():
            a.calc = calc
            en[name].append(a.get_potential_energy() / len(a))
    vols = np.array(vols)
    for name in en:
        en[name] = np.array(en[name]) - np.min(en[name])
    return vols, en


def fit_eos(V, E):
    """Parabolic fit -> (V0 [A^3/atom], E0 [eV/atom], B0 [GPa])."""
    a, b, c = np.polyfit(V, E, 2)
    V0 = -b / (2 * a)
    E0 = a * V0 ** 2 + b * V0 + c
    B0 = 2 * a * V0 * 160.21766208  # 1 eV/A^3 = 160.21766208 GPa
    return V0, E0, B0


print("run_eos / fit_eos defined — same physics as Section 9, now reusable.")

## 9b. Hands-on C+ — bulk modulus vs the DFT reference

Section 9 compared the three models against the **experimental** $B_0 \approx 99$ GPa.
Now compare against the **Materials Project DFT** value for the *same* structure,
which is the apples-to-apples reference (the models were trained on PBE labels).


In [ ]:
# --- Checkpoint B: model B0 vs MP DFT B0 ------------------------------------
B0_DFT = None
if si_mp_info.get("bulk_modulus_vrh") is not None:
    B0_DFT = si_mp_info["bulk_modulus_vrh"]
    print(f"MP DFT bulk modulus (VRH) for diamond Si: {B0_DFT:.1f} GPa")
else:
    # Fallback: the well-known PBE value for diamond Si if mp-api gave none.
    B0_DFT = 89.0
    print(f"No MP B0 returned; using literature PBE value B0_DFT = {B0_DFT:.1f} GPa")

B0_EXP = 99.0  # experimental diamond-Si bulk modulus (GPa)

vols_b, en_b = run_eos({
    "MACE-MP-0":    mace_calc,
    "SevenNet-0":   sevennet_calc,
    "MatterSim-1M": mattersim_calc,
})

print(f"\n{'Model':<15s} {'V0 (A^3/at)':>12s} {'B0 (GPa)':>10s} "
      f"{'|dB0|/B0_DFT':>13s} {'verdict':>12s}")
print("-" * 66)
baseline_B0 = {}
for name, e in en_b.items():
    V0, E0, B0 = fit_eos(vols_b, e)
    baseline_B0[name] = B0
    rel = abs(B0 - B0_DFT) / B0_DFT
    verdict = "usable" if rel < 0.15 else "off"   # Checkpoint B: <15% -> usable
    print(f"{name:<15s} {V0:>12.3f} {B0:>10.1f} {rel:>12.1%} {verdict:>12s}")
print("-" * 66)
print(f"{'MP DFT (VRH)':<15s} {si_mp_info.get('volume_per_atom', float('nan')):>12.3f} "
      f"{B0_DFT:>10.1f}")
print(f"{'Experiment':<15s} {20.0:>12.3f} {B0_EXP:>10.1f}")

print("\nCheckpoint B criterion: a model with |B0_model - B0_DFT|/B0_DFT < 15% "
      "is classified as 'usable' for elastic properties.")
print(f"Baseline MACE-MP-0 B0 = {baseline_B0['MACE-MP-0']:.1f} GPa "
      f"(we will try to improve this by fine-tuning).")

**Discussion**
- SevenNet and MatterSim are close to the experimental diamond-Si bulk modulus (~99 GPa), while this quick MACE-MP-0 EOS underestimates the curvature.
- Differences reflect the DFT functional used to label the training set, the fitted reference geometry, the model's inductive bias, and the limited unrelaxed EOS sweep used here.
- For any property prediction study, **always benchmark on a known reference before trusting predictions on new systems**.


## 10. Hands-on D — short NVT MD with MACE-MP

Run 200 fs of Langevin dynamics on a Si supercell at 1500 K. We log the instantaneous temperature and the potential energy per atom.

<!-- lecture13-visual:start:md -->
<div align="center">
  <img src="images/diagrams/md_thermostat.png" width="820"/>
  <br><em>In NVT dynamics, the MLP supplies forces while Langevin dynamics controls temperature.</em>
</div>
<!-- lecture13-visual:end:md -->



In [ ]:
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase import units as u

si_md = bulk("Si", "diamond", a=5.43, cubic=True).repeat((2, 2, 2))
si_md.calc = mace_calc
MaxwellBoltzmannDistribution(si_md, temperature_K=1500)

dyn = Langevin(
    si_md,
    timestep=1.0 * u.fs,
    temperature_K=1500,
    friction=0.01,
)

T_log, E_log = [], []
def record():
    T_log.append(si_md.get_temperature())
    E_log.append(si_md.get_potential_energy() / len(si_md))

dyn.attach(record, interval=1)

t0 = time.perf_counter()
dyn.run(200)
print(f"MD wall-time: {time.perf_counter() - t0:5.1f} s for 200 fs ({len(si_md)} atoms)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(T_log)
axes[0].axhline(1500, color="r", linestyle="--", alpha=0.5, label="target 1500 K")
axes[0].set_xlabel("Step (fs)")
axes[0].set_ylabel("Temperature (K)")
axes[0].set_title("Thermostat tracking")
axes[0].legend()

axes[1].plot(E_log)
axes[1].set_xlabel("Step (fs)")
axes[1].set_ylabel("E$_\\mathrm{pot}$ / atom  (eV)")
axes[1].set_title("Potential energy")

plt.tight_layout()

## 11. Hands-on E — fine-tuning MACE-MP-0 on Si

The benchmarks above showed that the *pretrained* MACE-MP-0 underestimates the
diamond-Si bulk modulus. The slides advertised **fine-tuning** as the fix, so
let us actually do it: continue training MACE-MP-0 on a small, targeted Si
dataset and re-measure $B_0$.

**The physicist's picture.** The foundation model is a good *prior* over the
PES of all inorganic chemistry. Fine-tuning is **Bayesian updating** with a few
hundred system-specific DFT observations — we nudge the prior toward the local
energy landscape of Si without throwing away the transferable features.

**Pipeline**
1. **Checkpoint C** — build `si_finetune.xyz`: ~100-300 rattled/strained Si
   configurations, each carrying an `energy` and a `forces` label.
2. **Checkpoint D** — run `mace_run_train --foundation_model=medium ...` and plot
   the training/validation learning curve.
3. **Checkpoint E** — load the fine-tuned model, re-run the EOS harness, and
   compare $B_0$ **before vs after** against the experimental 99 GPa.

> **Labels.** The cleanest labels are real DFT (e.g. an MPtrj Si slice queried via
> `mp-api`). To keep this notebook **self-contained and DFT-free**, we instead
> generate *pseudo-labels* with the strongest available foundation model
> (`mace_mp(model="large")`) — a standard *distillation* / *delta-learning*
> warm-up. Swap in real DFT labels (VASP/QE) for a production run.


### Checkpoint C — build the fine-tuning dataset `si_finetune.xyz`

Generate a diverse set of Si configurations and attach energy/force labels.
Diversity matters: we mix **isotropic volume strains** (to inform the bulk
modulus directly) with **random rattles** (to inform the local curvature).

> **Verification:** the written `.xyz` must contain **at least 100** structures,
> and every frame must carry an `energy` and per-atom `forces`.


In [ ]:
from ase.io import write as ase_write
from ase.calculators.singlepoint import SinglePointCalculator

# Stronger foundation model used ONLY as a pseudo-label generator.
# In a production run, replace this block with real DFT single-points.
from mace.calculators import mace_mp
label_calc = mace_mp(model="large", default_dtype="float32",
                     device=device, dispersion=False)

def build_si_finetune_set(n_strain: int = 9, n_rattle_per_strain: int = 14,
                          seed: int = 42):
    """Return a list of labelled ASE Atoms (energy + forces attached)."""
    rng = np.random.default_rng(seed)
    base = bulk("Si", "diamond", a=5.43, cubic=True).repeat((2, 2, 2))  # 64 atoms
    frames = []
    strains = np.linspace(0.95, 1.06, n_strain)   # isotropic volume strains
    for s in strains:
        for k in range(n_rattle_per_strain):
            a = base.copy()
            a.set_cell(a.cell * s, scale_atoms=True)
            if k > 0:                              # k==0: clean strained cell
                a.rattle(stdev=0.04, seed=int(rng.integers(0, 2**31)))
            a.calc = label_calc
            E = a.get_potential_energy()
            F = a.get_forces()
            # Freeze the labels onto the frame via a SinglePointCalculator.
            a.calc = SinglePointCalculator(a, energy=E, forces=F)
            a.info["energy"] = E
            frames.append(a)
    return frames

frames = build_si_finetune_set()
ase_write("si_finetune.xyz", frames, format="extxyz")

# --- Verification (Checkpoint C) -------------------------------------------
from ase.io import read as ase_read
check = ase_read("si_finetune.xyz", index=":")
n_ok = sum(
    1 for a in check
    if a.calc is not None
    and a.calc.results.get("energy") is not None
    and a.calc.results.get("forces") is not None
)
print(f"Wrote si_finetune.xyz with {len(check)} structures "
      f"({len(check[0])} atoms each).")
print(f"Frames carrying both energy & forces: {n_ok}")
assert len(check) >= 100, "Need >= 100 structures for Checkpoint C."
assert n_ok == len(check), "Every frame must carry energy + forces."
print("Checkpoint C PASS: >= 100 labelled Si configurations.")

### Checkpoint D — fine-tune with `mace_run_train`

`mace-torch` ships a CLI, `mace_run_train`, that continues training from a
foundation checkpoint when you pass `--foundation_model=medium`. The flags below
keep the medium architecture (8 radial bases, `max_ell=3`, 2 interaction layers)
and run a short fine-tune.

On a Colab **T4 GPU**, ~200 epochs over a few hundred 64-atom configs takes
roughly **5-10 minutes**. On CPU it is much slower — reduce `--max_num_epochs`.

> **Verification:** the validation energy MAE should show a **downward trend**
> within the first ~100 epochs (we parse it from the training log below).


In [ ]:
import subprocess, sys, glob, os

# We split si_finetune.xyz into train/valid via MACE's own --valid_fraction.
FT_NAME = "MACE_Si_finetune"
cmd = [
    "mace_run_train",
    "--name", FT_NAME,
    "--foundation_model", "medium",      # start from MACE-MP-0 medium
    "--train_file", "si_finetune.xyz",
    "--valid_fraction", "0.10",
    "--energy_key", "energy",
    "--forces_key", "forces",
    "--num_radial_basis", "8",
    "--max_ell", "3",
    "--num_interactions", "2",
    "--r_max", "5.0",
    "--batch_size", "4",
    "--valid_batch_size", "4",
    "--max_num_epochs", "200",
    "--swa",                              # stochastic weight averaging tail
    "--start_swa", "150",
    "--ema",
    "--default_dtype", "float32",
    "--device", device,
    "--seed", "42",
    "--save_cpu",
]
print("Running:\n  " + " ".join(cmd) + "\n")

# In Colab, run this cell directly. The training log streams to stdout.
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout[-4000:])      # tail of the log
if proc.returncode != 0:
    print("STDERR (tail):\n", proc.stderr[-2000:])

# The fine-tuned model is written as <FT_NAME>.model (and a *_compiled.model).
ft_models = sorted(glob.glob(f"{FT_NAME}*.model"))
print("\nFine-tuned model files:", ft_models)

In [ ]:
# --- Learning curve (Checkpoint D verification) ----------------------------
# MACE writes a results .txt / logs dir; parse the per-epoch validation metrics.
import json, glob

def load_mace_curve(name: str):
    """Return (epochs, valid_energy_mae_meV) parsed from MACE training logs."""
    epochs, val_e = [], []
    # MACE >=0.3 logs eval metrics as JSON lines under results/ or logs/.
    candidates = glob.glob(f"results/{name}*.txt") + glob.glob("logs/*.txt") \
                 + glob.glob(f"{name}*_train.txt")
    for path in candidates:
        with open(path) as fh:
            for line in fh:
                line = line.strip()
                if not line.startswith("{"):
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if rec.get("mode") == "eval" and "mae_e" in rec:
                    epochs.append(rec.get("epoch", len(epochs)))
                    # MACE logs energy MAE in eV; convert to meV/atom-ish display.
                    val_e.append(rec["mae_e"] * 1000.0)
        if epochs:
            break
    return np.array(epochs), np.array(val_e)

ep, val_e = load_mace_curve(FT_NAME)
if len(ep) > 1:
    plt.figure(figsize=(6.2, 4.0))
    plt.plot(ep, val_e, "o-", ms=3)
    plt.xlabel("epoch"); plt.ylabel("validation energy MAE (meV)")
    plt.title("MACE-MP-0 fine-tuning on Si — learning curve")
    plt.grid(alpha=0.3); plt.tight_layout()

    early = val_e[: max(1, len(val_e) // 2)].mean()
    late = val_e[len(val_e) // 2 :].mean()
    print(f"Mean val energy MAE: first half = {early:.2f} meV, "
          f"second half = {late:.2f} meV")
    print("Checkpoint D:", "PASS (downward trend)" if late < early
          else "CHECK — no clear decrease; train longer or check labels.")
else:
    print("Could not auto-parse the MACE log. Inspect results/*.txt manually; "
          "the validation energy MAE should trend down within ~100 epochs.")

### Checkpoint E — re-measure the bulk modulus (before vs after)

Load the fine-tuned MACE model as an ASE calculator, run the **same** EOS harness,
and overlay the before/after curves. The acceptance criterion is physical: the
fine-tuned $B_0$ should sit **closer to the experimental 99 GPa** than the
pretrained baseline (~78.8 GPa).


In [ ]:
# --- Load the fine-tuned model as an ASE calculator -------------------------
from mace.calculators import MACECalculator

ft_path = ft_models[0] if ft_models else None
if ft_path is None:
    raise FileNotFoundError(
        "No fine-tuned model found — run the Checkpoint D cell first.")

mace_ft_calc = MACECalculator(model_paths=[ft_path],
                              device=device, default_dtype="float32")
print(f"Loaded fine-tuned calculator from {ft_path}")

# --- Re-run the EOS harness with baseline + fine-tuned models ---------------
vols_e, en_e = run_eos({
    "MACE-MP-0 (pretrained)": mace_calc,
    "MACE-MP-0 (fine-tuned)": mace_ft_calc,
})

V0_pre, _, B0_pre = fit_eos(vols_e, en_e["MACE-MP-0 (pretrained)"])
V0_ft,  _, B0_ft  = fit_eos(vols_e, en_e["MACE-MP-0 (fine-tuned)"])

print(f"\n{'Model':<24s} {'V0 (A^3/at)':>12s} {'B0 (GPa)':>10s}")
print("-" * 48)
print(f"{'MACE-MP-0 (pretrained)':<24s} {V0_pre:>12.3f} {B0_pre:>10.1f}")
print(f"{'MACE-MP-0 (fine-tuned)':<24s} {V0_ft:>12.3f} {B0_ft:>10.1f}")
print("-" * 48)
print(f"{'Experiment':<24s} {20.0:>12.3f} {99.0:>10.1f}")

# --- Verification (Checkpoint E) -------------------------------------------
B0_EXP = 99.0
err_pre = abs(B0_pre - B0_EXP)
err_ft = abs(B0_ft - B0_EXP)
print(f"\n|B0 - 99| : pretrained = {err_pre:.1f} GPa, "
      f"fine-tuned = {err_ft:.1f} GPa")
print("Checkpoint E:",
      "PASS — fine-tuning moved B0 toward experiment."
      if err_ft < err_pre else
      "CHECK — B0 did not improve; try more/better-distributed labels or epochs.")

In [ ]:
# --- Before/after EOS overlay ----------------------------------------------
plt.figure(figsize=(6.5, 4.2))
plt.plot(vols_e, en_e["MACE-MP-0 (pretrained)"], "o-",
         label=f"pretrained  (B0={B0_pre:.0f} GPa)")
plt.plot(vols_e, en_e["MACE-MP-0 (fine-tuned)"], "s-",
         label=f"fine-tuned  (B0={B0_ft:.0f} GPa)")
plt.xlabel("Volume per atom (A^3)")
plt.ylabel(r"E - E$_\mathrm{min}$  (eV/atom)")
plt.title("Bulk Si EOS — MACE-MP-0 before vs after fine-tuning")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()

## 12. When to use which?

- **MatterSim** — broad $(T, P)$ coverage out of the box: high-pressure phase diagrams, hot liquids, melting curves. Largest training set among open foundation MLPs.
- **MACE-MP-0** — strong general-purpose baseline for inorganic structures near equilibrium. Cheap inference. Active community.
- **MACE-OFF** — drug-like organic molecules; the natural choice for biomolecular MD.
- **SevenNet** — large-scale parallel MD ($> 10^5$ atoms). The built-in ZBL repulsion keeps it stable under aggressive perturbations or high pressure.
- **Allegro** — train per-system for *production accuracy* once a foundation model has bootstrapped the dataset.

**Recommended workflow for a new project**
1. Sanity-check several foundation MLPs against DFT on a handful of representative configurations.
2. If accuracy is sufficient, use the best one as-is.
3. Otherwise, **fine-tune** on a small DFT dataset; active learning helps select informative structures.
4. For extreme accuracy or unusual chemistry, train a dedicated NequIP/MACE/Allegro model from scratch.

<!-- lecture13-visual:start:workflow -->
<table style="width:100%; border:0;">
<tr>
<td align="center" style="border:0; padding:8px;"><img src="images/ai/05_active_learning_loop.webp" width="360"/><br><em>Active learning loop: simulate, flag uncertainty, label with DFT, retrain.</em></td>
<td align="center" style="border:0; padding:8px;"><img src="images/ai/06_production_md.webp" width="360"/><br><em>Production deployment: large boxes, domain decomposition, and GPU-parallel inference.</em></td>
</tr>
</table>
<!-- lecture13-visual:end:workflow -->


**From scratch vs fine-tune a foundation model — how to choose (by design)**

This course deliberately teaches *both* paths so you can pick the right one:

| You have… | Best path | Why |
|---|---|---|
| 10²–10³ DFT structures, a *common* chemistry, near-equilibrium targets | **Fine-tune** MACE-MP-0 / SevenNet (this section) | The foundation prior already covers the chemistry; a few hundred labels correct the local PES cheaply. |
| 10³–10⁴ DFT structures, exotic chemistry or extreme $(T,P)$ | **Fine-tune, then active-learn** | Start from the prior, then grow the dataset where the model is uncertain. |
| 10⁴⁺ DFT structures, a single high-value system, accuracy is paramount | **Train from scratch** (NequIP / MACE / Allegro) | No transfer bias; the architecture can be tuned to the system and squeeze out the last meV/atom. |
| Almost no data, just need a quick screen | **Use a foundation model as-is** | Zero training cost; validate against a few DFT points first (Sections 8b/9b). |

The rule of thumb: **fine-tuning trades a little peak accuracy for a 10–100×
reduction in DFT cost**, which is usually the right trade unless you are
publishing benchmark-grade numbers for one specific material.


## 13. References

1. Batzner *et al.*, *Nat. Commun.* **13**, 2453 (2022). **NequIP**.
2. Batatia *et al.*, *NeurIPS* (2022); arXiv:2206.07697. **MACE**.
3. Batatia *et al.*, arXiv:2401.00096 (2024). **MACE-MP**.
4. Park *et al.*, *J. Chem. Theory Comput.* (2024). **SevenNet**.
5. Yang *et al.*, arXiv:2405.04967 (2024). **MatterSim**.
6. Drautz, *Phys. Rev. B* **99**, 014104 (2019). **Atomic Cluster Expansion**.
7. Musaelian *et al.*, *Nat. Commun.* **14**, 579 (2023). **Allegro**.
8. Deng *et al.*, *Nat. Mach. Intell.* (2023). **CHGNet**.
9. Chen & Ong, *Nat. Comput. Sci.* **2**, 718 (2022). **M3GNet**.

---

*End of Lecture 13.*
